In [2]:
# ==========================================
# 1. INSTALAR LIBRERIAS
# ==========================================
!pip install pandas requests beautifulsoup4 lxml openpyxl unidecode -q

# ==========================================
# 2. IMPORTAR LIBRERIAS
# ==========================================
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from unidecode import unidecode

# ==========================================
# 3. FUNCION PARA LIMPIAR NOMBRES DE COLUMNAS
# ==========================================
def limpiar_columnas(cols):
    nuevas = []
    for c in cols:
        c = str(c)
        c = unidecode(c)
        c = c.strip().lower()
        c = c.replace(" ", "_")
        c = c.replace("-", "_")
        c = c.replace(".", "")
        c = c.replace("/", "_")
        nuevas.append(c)
    return nuevas

# ==========================================
# 4. FUENTE 1. VIOLENCIA INTRAFAMILIAR
# ==========================================
# OPCION A
# Si el archivo ya está subido a Colab, usa el nombre exacto del archivo
ruta_violencia = "Reporte_Delito_Violencia_Intrafamiliar_Policía_Nacional_20260407.csv"

df_violencia = pd.read_csv(ruta_violencia)
df_violencia.columns = limpiar_columnas(df_violencia.columns)

print("FUENTE 1. Violencia Intrafamiliar")
print("Dimensiones:", df_violencia.shape)
print("Columnas:")
print(df_violencia.columns.tolist())
display(df_violencia.head())

# ==========================================
# 5. TRANSFORMACION BASICA FUENTE 1
# ==========================================
# Ajusta estos nombres si en tu archivo aparecen un poco distinto
mapa_esperado = {
    "fecha_hecho": "fecha_hecho",
    "departamento": "departamento",
    "municipio": "municipio",
    "genero": "genero",
    "grupo_etario": "grupo_etario",
    "cantidad": "cantidad"
}

for col in ["departamento", "municipio", "genero", "grupo_etario"]:
    if col in df_violencia.columns:
        df_violencia[col] = df_violencia[col].astype(str).str.upper().str.strip()

if "fecha_hecho" in df_violencia.columns:
    df_violencia["fecha_hecho"] = pd.to_datetime(df_violencia["fecha_hecho"], errors="coerce", dayfirst=True)
    df_violencia["anio"] = df_violencia["fecha_hecho"].dt.year

if "cantidad" in df_violencia.columns:
    df_violencia["cantidad"] = pd.to_numeric(df_violencia["cantidad"], errors="coerce")

print("\nTransformación básica de violencia lista")
display(df_violencia.head())

# ==========================================
# 6. FUENTE 2. DANE IPM POR DEPARTAMENTO
# ==========================================
url_dane = "https://www.dane.gov.co/index.php/estadisticas-por-tema/pobreza-y-condiciones-de-vida/pobreza-multidimensional"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url_dane, headers=headers)
soup = BeautifulSoup(response.text, "lxml")

excel_links = []
for a in soup.find_all("a", href=True):
    href = a["href"]
    full_url = urljoin(url_dane, href)
    if full_url.endswith(".xls") or full_url.endswith(".xlsx"):
        excel_links.append(full_url)

print("Archivos Excel encontrados:")
for link in excel_links:
    print(link)

# Buscar el archivo departamental
link_ipm = None
for link in excel_links:
    if "departamental" in link.lower() and ("pmultidimensional" in link.lower() or "pobreza" in link.lower()):
        link_ipm = link
        break

print("\nArchivo seleccionado:")
print(link_ipm)

nombre_archivo = "ipm_dane_departamental.xlsx"
r = requests.get(link_ipm, headers=headers)
with open(nombre_archivo, "wb") as f:
    f.write(r.content)

# Leer estructura cruda
hoja_objetivo = "IPM_Departamentos"
df_raw = pd.read_excel(nombre_archivo, sheet_name=hoja_objetivo, header=None)

print("\nFUENTE 2. DANE IPM")
print("Dimensiones crudas:", df_raw.shape)
display(df_raw.head(15))

# Reconstruir encabezados
fila_anios = df_raw.iloc[11].tolist()
fila_sub = df_raw.iloc[12].tolist()

columnas = []
for i in range(len(df_raw.columns)):
    anio = fila_anios[i]
    sub = fila_sub[i]

    if i == 0:
        columnas.append("departamento")
    else:
        anio_str = str(anio).replace("**", "").strip() if pd.notna(anio) else ""
        sub_str = str(sub).strip() if pd.notna(sub) else ""
        columnas.append(f"{anio_str}_{sub_str}")

df_dane = df_raw.iloc[13:].copy()
df_dane.columns = columnas
df_dane = df_dane.reset_index(drop=True)

# Filtrar columnas Total
columnas_utiles = ["departamento"]
for col in df_dane.columns:
    if "Total" in col:
        columnas_utiles.append(col)

df_dane = df_dane[columnas_utiles]

# Renombrar columnas de años
rename_dict = {}
for col in df_dane.columns:
    if col != "departamento":
        anio = col.replace("_Total", "").replace("*", "").strip()
        rename_dict[col] = anio

df_dane = df_dane.rename(columns=rename_dict)

# Pasar a formato largo
df_dane_long = df_dane.melt(
    id_vars="departamento",
    var_name="anio",
    value_name="ipm_total"
)

df_dane_long["departamento"] = df_dane_long["departamento"].astype(str).str.upper().str.strip()
df_dane_long["anio"] = pd.to_numeric(df_dane_long["anio"], errors="coerce")
df_dane_long["ipm_total"] = pd.to_numeric(df_dane_long["ipm_total"], errors="coerce")
df_dane_long = df_dane_long.dropna().reset_index(drop=True)

print("\nDANE transformado")
print("Dimensiones:", df_dane_long.shape)
display(df_dane_long.head())

# ==========================================
# 7. FUENTE 3. API PUBLICA DIVIPOLA
# ==========================================
api_url = "https://www.datos.gov.co/resource/gdxc-w37w.json?$limit=5000"

resp = requests.get(api_url, headers={"User-Agent": "Mozilla/5.0"})
resp.raise_for_status()

data_api = resp.json()
df_api = pd.DataFrame(data_api)
df_api.columns = limpiar_columnas(df_api.columns)

print("\nFUENTE 3. API DIVIPOLA")
print("Dimensiones:", df_api.shape)
print("Columnas:")
print(df_api.columns.tolist())
display(df_api.head())

# Asignación directa de columnas correctas
col_cod_dpto = "cod_dpto"
col_depto = "dpto"
col_cod_mpio = "cod_mpio"
col_mpio = "nom_mpio"
col_tipo = "tipo_municipio"
col_longitud = "longitud"
col_latitud = "latitud"

# Verificar que sí existan
columnas_requeridas = [
    col_cod_dpto, col_depto, col_cod_mpio, col_mpio,
    col_tipo, col_longitud, col_latitud
]

faltantes = [c for c in columnas_requeridas if c not in df_api.columns]
if faltantes:
    raise ValueError(f"Faltan columnas en la API: {faltantes}")

# Crear tabla geográfica limpia
df_geo = df_api[columnas_requeridas].copy()

# Estandarizar texto
df_geo[col_depto] = df_geo[col_depto].astype(str).str.upper().str.strip()
df_geo[col_mpio] = df_geo[col_mpio].astype(str).str.upper().str.strip()
df_geo[col_tipo] = df_geo[col_tipo].astype(str).str.upper().str.strip()

# Convertir coordenadas de coma a punto
df_geo[col_longitud] = (
    df_geo[col_longitud]
    .astype(str)
    .str.replace(",", ".", regex=False)
)
df_geo[col_latitud] = (
    df_geo[col_latitud]
    .astype(str)
    .str.replace(",", ".", regex=False)
)

df_geo[col_longitud] = pd.to_numeric(df_geo[col_longitud], errors="coerce")
df_geo[col_latitud] = pd.to_numeric(df_geo[col_latitud], errors="coerce")

# Quitar duplicados
df_geo = df_geo.drop_duplicates().reset_index(drop=True)

print("\nTabla geográfica depurada")
print("Dimensiones:", df_geo.shape)
display(df_geo.head())

# Guardar resultado
df_geo.to_csv("03_api_divipola.csv", index=False)
print("\nArchivo guardado correctamente: 03_api_divipola.csv")

# ==========================================
# 8. GUARDAR RESULTADOS INTERMEDIOS
# ==========================================
df_violencia.to_csv("01_violencia_raw_limpia.csv", index=False)
df_dane_long.to_csv("02_dane_ipm_long.csv", index=False)
df_geo.to_csv("03_api_divipola.csv", index=False)

print("\nArchivos guardados correctamente:")
print("01_violencia_raw_limpia.csv")
print("02_dane_ipm_long.csv")
print("03_api_divipola.csv")

FUENTE 1. Violencia Intrafamiliar
Dimensiones: (660756, 8)
Columnas:
['departamento', 'municipio', 'codigo_dane', 'armas_medios', 'fecha_hecho', 'genero', 'grupo_etario', 'cantidad']


,departamento,municipio,codigo_dane,armas_medios,fecha_hecho,genero,grupo_etario,cantidad
0,ANTIOQUIA,Medellín (CT),5001000,SIN EMPLEO DE ARMAS,18/09/2025,MASCULINO,MENORES,1
1,CUNDINAMARCA,Bogotá D.C. (CT),11001000,SIN EMPLEO DE ARMAS,23/10/2025,FEMENINO,MENORES,5
2,CUNDINAMARCA,Bogotá D.C. (CT),11001000,SIN EMPLEO DE ARMAS,23/10/2025,MASCULINO,MENORES,5
3,CAUCA,Cajibío,19130000,SIN EMPLEO DE ARMAS,23/10/2025,FEMENINO,ADULTOS,1
4,NORTE DE SANTANDER,Teorama,54800000,SIN EMPLEO DE ARMAS,06/10/2025,MASCULINO,ADULTOS,1



Transformación básica de violencia lista


,departamento,municipio,codigo_dane,armas_medios,fecha_hecho,genero,grupo_etario,cantidad,anio
0,ANTIOQUIA,MEDELLÍN (CT),5001000,SIN EMPLEO DE ARMAS,2025-09-18,MASCULINO,MENORES,1,2025
1,CUNDINAMARCA,BOGOTÁ D.C. (CT),11001000,SIN EMPLEO DE ARMAS,2025-10-23,FEMENINO,MENORES,5,2025
2,CUNDINAMARCA,BOGOTÁ D.C. (CT),11001000,SIN EMPLEO DE ARMAS,2025-10-23,MASCULINO,MENORES,5,2025
3,CAUCA,CAJIBÍO,19130000,SIN EMPLEO DE ARMAS,2025-10-23,FEMENINO,ADULTOS,1,2025
4,NORTE DE SANTANDER,TEORAMA,54800000,SIN EMPLEO DE ARMAS,2025-10-06,MASCULINO,ADULTOS,1,2025


Archivos Excel encontrados:
https://www.dane.gov.co/files/Transparencia/Registro-de-activos-de-informacion.xlsx
https://www.dane.gov.co/files/operaciones/PM/anex-PMultidimensional-2024.xlsx
https://www.dane.gov.co/files/operaciones/PM/anex-PMultidimensional-Departamental-2024.xlsx
https://www.dane.gov.co/files/operaciones/PM/anex-PMPDET-2024.xlsx
https://www.dane.gov.co/files/operaciones/PM/anex-PM-PoblacionVictima-2024.xlsx

Archivo seleccionado:
https://www.dane.gov.co/files/operaciones/PM/anex-PMultidimensional-Departamental-2024.xlsx

FUENTE 2. DANE IPM
Dimensiones crudas: (54, 22)


...


FUENTE 3. API DIVIPOLA
Dimensiones: (1122, 7)
Columnas:
['cod_dpto', 'dpto', 'cod_mpio', 'nom_mpio', 'tipo_municipio', 'longitud', 'latitud']


,cod_dpto,dpto,cod_mpio,nom_mpio,tipo_municipio,longitud,latitud
0,05,ANTIOQUIA,05001,MEDELLÍN,Municipio,"-75,581775","6,246631"
1,05,ANTIOQUIA,05002,ABEJORRAL,Municipio,"-75,428739","5,789315"
2,05,ANTIOQUIA,05004,ABRIAQUÍ,Municipio,"-76,064304","6,632282"
3,05,ANTIOQUIA,05021,ALEJANDRÍA,Municipio,"-75,141346","6,376061"
4,05,ANTIOQUIA,05030,AMAGÁ,Municipio,"-75,702188","6,038708"



Tabla geográfica depurada
Dimensiones: (1122, 7)


,cod_dpto,dpto,cod_mpio,nom_mpio,tipo_municipio,longitud,latitud
0,05,ANTIOQUIA,05001,MEDELLÍN,MUNICIPIO,-75.581775,6.246631
1,05,ANTIOQUIA,05002,ABEJORRAL,MUNICIPIO,-75.428739,5.789315
2,05,ANTIOQUIA,05004,ABRIAQUÍ,MUNICIPIO,-76.064304,6.632282
3,05,ANTIOQUIA,05021,ALEJANDRÍA,MUNICIPIO,-75.141346,6.376061
4,05,ANTIOQUIA,05030,AMAGÁ,MUNICIPIO,-75.702188,6.038708



Archivo guardado correctamente: 03_api_divipola.csv

Archivos guardados correctamente:
01_violencia_raw_limpia.csv
02_dane_ipm_long.csv
03_api_divipola.csv
